# BigQuant single-factor submission

This notebook is generated for exactly one factor submission. Upload this `.ipynb` file alone.


In [2]:
def main(datasources, start_date, end_date):
    """Return one AutoMiner v2 auction/liquidity-recovery factor."""

    import json
    import numpy as np
    import pandas as pd
    try:
        import dai
    except Exception:
        dai = None

    cfg = json.loads('{"expected_failure_mode": "late recovery signal is swallowed by residual volatility exposure", "factor_id": "amv2_early_pressure_late_recovery_quality_5d9e0b41", "mechanism_family": "EARLY_PRESSURE_LATE_RECOVERY_QUALITY", "meta_pattern": "tension", "signal_sources": ["bar1m.early_return_pressure", "bar1m.late_return_recovery", "financial.cash_profit_quality", "exposure.RESVOL"]}')
    family = cfg["mechanism_family"]
    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()
    left_ts = start_ts - pd.DateOffset(years=3)
    right_ts = end_ts + pd.Timedelta(days=1)

    def _resolve(names, default):
        if isinstance(datasources, dict):
            for name in names:
                value = datasources.get(name)
                if value is not None:
                    return value
        return default

    def _query(names, default_table, fields, left, right):
        source = _resolve(names, default_table)
        if isinstance(source, pd.DataFrame):
            frame = source.copy()
        elif hasattr(source, "query") and not isinstance(source, str):
            try:
                frame = source.query("SELECT " + ", ".join(fields), filters={"date": [left, right]}).df()
            except Exception:
                frame = pd.DataFrame()
        elif dai is not None:
            try:
                frame = dai.query(
                    "SELECT " + ", ".join(fields) + " FROM " + str(source),
                    filters={"date": [left, right]},
                    compression=True,
                ).df()
            except Exception:
                frame = pd.DataFrame()
        else:
            frame = pd.DataFrame()
        if frame.empty:
            return frame
        frame = frame.copy()
        if "date" in frame.columns:
            frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
            frame = frame[(frame["date"] >= pd.Timestamp(left)) & (frame["date"] < pd.Timestamp(right))]
        if "instrument" in frame.columns:
            frame["instrument"] = frame["instrument"].astype(str)
        return frame.dropna(subset=[c for c in ["date", "instrument"] if c in frame.columns])

    def _num(frame, column, default=np.nan):
        if column not in frame.columns:
            return pd.Series(default, index=frame.index, dtype=float)
        return pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

    def _rank(frame, values):
        values = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
        return values.groupby(frame["date"], sort=False).rank(pct=True).fillna(0.5)

    def _rank_z(frame, values):
        return (_rank(frame, values) - 0.5) * 2.0

    query_left = left_ts.strftime("%Y-%m-%d")
    query_start = start_ts.strftime("%Y-%m-%d")
    query_end = right_ts.strftime("%Y-%m-%d")

    instruments = _query(
        ["instruments", "instrument", "bigalpha_2026_instruments"],
        "bigalpha_2026_instruments",
        ["date", "instrument"],
        query_start,
        query_end,
    )
    if instruments.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    instruments["date"] = pd.to_datetime(instruments["date"], errors="coerce").dt.normalize()
    panel = instruments.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"]).copy()

    bar = _query(
        ["bar1m", "stock_bar1m", "bigalpha_2026_stock_bar1m"],
        "bigalpha_2026_stock_bar1m",
        ["date", "instrument", "open", "high", "low", "close", "amount", "volume", "deal_number"],
        query_start,
        query_end,
    )
    if bar.empty:
        for column in ["opening_gap", "first_amount_share", "reclaim", "early_pressure", "late_recovery", "path_range", "dryup"]:
            panel[column] = 0.0
    else:
        bar = bar.sort_values(["instrument", "date"]).copy()
        bar["day"] = pd.to_datetime(bar["date"], errors="coerce").dt.normalize()
        bar["instrument"] = bar["instrument"].astype(str)
        for column in ["open", "high", "low", "close"]:
            bar[column] = _num(bar, column, np.nan)
        bar["amount"] = _num(bar, "amount", 0.0).fillna(0.0).clip(lower=0.0)
        g = bar.groupby(["day", "instrument"], sort=False)
        bar["slot"] = g.cumcount()
        bar["slot_n"] = g["slot"].transform("max").replace(0, 1)
        first = bar["slot"] <= (bar["slot_n"] / 4.0)
        late = bar["slot"] > (bar["slot_n"] * 3.0 / 4.0)
        keys = ["day", "instrument"]

        daily = g.agg(
            open_first=("open", "first"),
            close_last=("close", "last"),
            high_max=("high", "max"),
            low_min=("low", "min"),
            amount_sum=("amount", "sum"),
        ).reset_index()

        def _sum(mask, value_col):
            work = bar.loc[mask, keys + [value_col]].copy()
            return work.groupby(keys, sort=False)[value_col].sum(min_count=1).rename(value_col)

        def _first(mask, value_col):
            work = bar.loc[mask, keys + [value_col]].dropna(subset=[value_col]).copy()
            return work.groupby(keys, sort=False)[value_col].first().rename(value_col)

        def _last(mask, value_col):
            work = bar.loc[mask, keys + [value_col]].dropna(subset=[value_col]).copy()
            return work.groupby(keys, sort=False)[value_col].last().rename(value_col)

        daily = daily.join(_sum(first, "amount").rename("first_amount"), on=keys)
        daily = daily.join(_sum(late, "amount").rename("late_amount"), on=keys)
        daily = daily.join(_first(first, "open").rename("first_open"), on=keys)
        daily = daily.join(_last(first, "close").rename("first_close"), on=keys)
        daily = daily.join(_first(late, "open").rename("late_open"), on=keys)
        daily = daily.join(_last(late, "close").rename("late_close"), on=keys)
        denom = daily["amount_sum"].replace(0, np.nan)
        daily["first_amount_share"] = (daily["first_amount"] / denom).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        daily["dryup"] = 1.0 - daily["first_amount_share"].clip(0.0, 1.0)
        daily["path_range"] = ((daily["high_max"] - daily["low_min"]) / daily["open_first"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        daily["reclaim"] = ((daily["close_last"] - daily["low_min"]) / (daily["high_max"] - daily["low_min"]).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0.5)
        daily["early_pressure"] = (daily["first_close"] / daily["first_open"].replace(0, np.nan) - 1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        daily["late_recovery"] = (daily["late_close"] / daily["late_open"].replace(0, np.nan) - 1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        daily["date"] = daily["day"]
        daily = daily.sort_values(["instrument", "date"])
        prev_close = daily.groupby("instrument", sort=False)["close_last"].shift(1)
        daily["opening_gap"] = (daily["open_first"] / prev_close.replace(0, np.nan) - 1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        panel = panel.merge(
            daily[["date", "instrument", "opening_gap", "first_amount_share", "dryup", "path_range", "reclaim", "early_pressure", "late_recovery"]],
            on=["date", "instrument"],
            how="left",
        )
        for column in ["opening_gap", "first_amount_share", "dryup", "path_range", "reclaim", "early_pressure", "late_recovery"]:
            panel[column] = pd.to_numeric(panel.get(column, 0.0), errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    exp = _query(
        ["exposure", "risk_exposure", "bigalpha_2026_exposure"],
        "bigalpha_2026_exposure",
        ["date", "instrument", "RESVOL", "LIQUIDTY", "BTOP", "SIZE"],
        query_start,
        query_end,
    )
    if not exp.empty:
        exp["date"] = pd.to_datetime(exp["date"], errors="coerce").dt.normalize()
        panel = panel.merge(exp.drop_duplicates(["date", "instrument"]), on=["date", "instrument"], how="left")
    for column in ["RESVOL", "LIQUIDTY", "BTOP", "SIZE"]:
        panel[column] = pd.to_numeric(panel.get(column, 0.0), errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    fin = _query(
        ["financial", "financial_statement", "bigalpha_2026_financial"],
        "bigalpha_2026_financial",
        ["date", "instrument", "net_cffoa", "net_profit", "total_assets", "cash_received_from_sales_and_services", "operating_revenue", "total_operating_revenue"],
        query_left,
        query_end,
    )
    if fin.empty:
        panel["quality"] = 0.0
        panel["sales_cash"] = 0.0
        panel["report_age"] = 365.0
    else:
        fin = fin.sort_values(["instrument", "date"]).copy()
        fin["date"] = pd.to_datetime(fin["date"], errors="coerce").dt.normalize()
        assets = _num(fin, "total_assets", np.nan).where(lambda s: s.abs() > 1e-12, np.nan)
        revenue = _num(fin, "operating_revenue", np.nan)
        if revenue.isna().all() and "total_operating_revenue" in fin.columns:
            revenue = _num(fin, "total_operating_revenue", np.nan)
        revenue = revenue.where(revenue.abs() > 1e-12, np.nan)
        fin["quality"] = ((_num(fin, "net_cffoa") - _num(fin, "net_profit")) / assets).replace([np.inf, -np.inf], np.nan).clip(-25, 25)
        fin["sales_cash"] = (_num(fin, "cash_received_from_sales_and_services") / revenue).replace([np.inf, -np.inf], np.nan).clip(-25, 25)
        fin["financial_date"] = fin["date"]
        keep = fin[["date", "instrument", "financial_date", "quality", "sales_cash"]].drop_duplicates(["instrument", "date"], keep="last")
        pieces = []
        for instrument, left in panel.groupby("instrument", sort=False):
            right = keep[keep["instrument"] == instrument].sort_values("date")
            if right.empty:
                pieces.append(left.copy())
            else:
                pieces.append(pd.merge_asof(left.sort_values("date"), right, on="date", by="instrument", direction="backward"))
        panel = pd.concat(pieces, ignore_index=True)
        panel["report_age"] = (panel["date"] - panel.get("financial_date", panel["date"])).dt.days.clip(lower=0, upper=720)
        for column in ["quality", "sales_cash"]:
            panel[column] = pd.to_numeric(panel.get(column, 0.0), errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    quality = _rank_z(panel, panel["quality"])
    sales_cash = _rank_z(panel, panel["sales_cash"])
    freshness = 1.0 - _rank(panel, panel["report_age"])
    gap = _rank_z(panel, panel["opening_gap"])
    abs_gap = _rank_z(panel, panel["opening_gap"].abs())
    first_liq = _rank_z(panel, panel["first_amount_share"])
    dryup = _rank_z(panel, panel["dryup"])
    reclaim = _rank_z(panel, panel["reclaim"])
    early = _rank_z(panel, panel["early_pressure"])
    late = _rank_z(panel, panel["late_recovery"])
    path = _rank_z(panel, panel["path_range"])
    resvol = _rank_z(panel, panel["RESVOL"])
    liquidity = _rank_z(panel, panel["LIQUIDTY"])
    btop = _rank_z(panel, panel["BTOP"])
    size = _rank_z(panel, panel["SIZE"])

    if family == "OPENING_GAP_LIQUIDITY_RECLAIM":
        raw = reclaim * (0.5 - abs_gap.abs()) + 0.35 * first_liq - 0.30 * liquidity - 0.15 * resvol
    elif family == "EARLY_PRESSURE_LATE_RECOVERY_QUALITY":
        raw = quality * (late - early) + 0.25 * reclaim - 0.30 * resvol
    elif family == "AUCTION_CASHFLOW_CONFIRMATION":
        raw = sales_cash * first_liq + 0.30 * reclaim - 0.20 * size
    elif family == "LIQUIDITY_DRYUP_RECOVERY_SEQUENCE":
        seq = (dryup + late + reclaim).groupby(panel["instrument"], sort=False).transform(lambda x: x.rolling(4, min_periods=1).mean())
        raw = seq - 0.35 * liquidity - 0.25 * resvol
    elif family == "REPORT_FRESH_OPENING_REVERSAL_CALENDAR":
        raw = freshness * (-gap + reclaim) + 0.20 * quality - 0.25 * btop
    else:
        raw = reclaim + 0.25 * quality - 0.20 * resvol

    raw = pd.to_numeric(raw, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if int(raw.nunique(dropna=True)) <= 1:
        raw = reclaim + 0.25 * quality + 0.15 * sales_cash - 0.20 * resvol
    panel["factor"] = _rank(panel, raw).replace([np.inf, -np.inf], np.nan).fillna(0.5)
    out = panel.loc[(panel["date"] >= start_ts) & (panel["date"] <= end_ts), ["date", "instrument", "factor"]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.5)
    return out.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"], keep="last").sort_values(["date", "instrument"]).reset_index(drop=True)
